# 앙상블 실험 — 베이스 + v2 어댑터 표 합치기

## 규칙 확인 (주최 측 답변, 2026-08-23)

> 베이스 모델이 지정된 Qwen2.5-3B-Instruct로 동일한 이상, 그 위에 여러 LoRA 어댑터를 학습하여
> 추론 시 앙상블하는 것은 가능함. 다만 **유형별로 세분화된 전문 어댑터를 다수 제작하여 라우팅**하는 등
> 사실상 복수의 특화 모델을 구축하는 수준에 이르는 경우 대회 취지에 부합하지 않을 수 있음.

| 우려 사항 | 이 실험 |
|---|---|
| 어댑터 다수 | **1개** (v2) |
| 유형별 라우팅 | **없음** — 모든 문제에 동일 적용 |
| 특화 모델 구축 | 없음 — 단일 어댑터 균일 사용 |

**허용 범위 안에서도 가장 단순한 형태**입니다.

## 왜 이걸 하나

두 모델의 강점이 다릅니다.

| | 베이스 | v2 |
|---|---|---|
| 정답을 찾는 능력 (pass@32) | 0.8633 | **0.8733** |
| 표를 모으는 능력 (득표율) | **0.726** | 0.650 |
| 결과 (maj@32) | **0.7400** | 0.7300 |

베이스는 **집중력**이 좋고, v2는 **새로 푸는 문제**가 있습니다.
표를 합치면 베이스가 표를 모으는 축 역할을 하고, v2가 새로 푸는 문제를 보탤 수 있습니다.

## 이번에 보는 것

한 세션에서 베이스 32표 + v2 32표를 뽑고, **이미 생성한 결과를 여러 조합으로 다시 묶어** 비교합니다.
조합 계산은 GPU를 전혀 쓰지 않으므로 공짜입니다.

- `base32` / `v2_32` — 단독 (재현 확인용)
- `base32+v2_32` — 1:1 합산
- `base32+v2_16`, `base16+v2_32` — 비율 조정
- **`pass@64`** — 두 모델의 합집합 커버리지. 0.8733보다 높으면 서로 다른 문제를 푼다는 뜻

## 소요: 약 1시간 45분


---
## [1] 설정 ▶️

In [ ]:
N_SAMPLES  = 32        # 각 모델당
TEMP       = 0.8       # 0.7과 결과가 동일했으므로 0.8 유지
MAX_TOKENS = 1024
VALID_N    = 300       # 절대 변경 금지
SEED       = 42        # 절대 변경 금지
LORA_RANK  = 64        # v2 어댑터
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
print(f"{VALID_N}문제 x {N_SAMPLES}샘플 x 2(base+v2) = {VALID_N*N_SAMPLES*2:,} 생성")

---
## [2] vLLM 설치 ⏭️

In [ ]:
!pip install -q -U vllm 2>&1 | tail -3

---
## [3] protobuf ⏭️
---
## ⛔ Restart Session → [1]부터
---

In [ ]:
!pip install -q -U "protobuf>=6.33.6,<7" 2>&1 | tail -2
import google.protobuf as p
print("protobuf", p.__version__); assert p.__version__.startswith("6.")

---
## [4] 데이터 + 어댑터 ▶️

⚠️ Input에 **어댑터 Dataset은 `qwen25-3b-v2-lora` 하나만** 붙어 있어야 합니다.
다른 어댑터가 같이 있으면 자동 탐색이 엉뚱한 걸 잡습니다.

In [ ]:
import glob, os, pandas as pd

def find_csv(must_have, must_not=()):
    for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
        b = os.path.basename(p).lower()
        if all(k in b for k in must_have) and not any(k in b for k in must_not):
            return p

train = pd.read_csv(find_csv(["train"], must_not=["filtered","ids","leaderboard","test"]))
train = train[~train["id"].isin(set(pd.read_csv(find_csv(["filtered","ids"]))["id"]))].reset_index(drop=True)
assert len(train) == 16373

work = train.sample(VALID_N, random_state=SEED).reset_index(drop=True)
gold = work["answer"].tolist()
print(f"{len(work)}문제 | 첫 id: {work.iloc[0]['id']}  (train-004925 여야 함)")

cfgs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert len(cfgs) == 1, f"어댑터가 {len(cfgs)}개 발견됨. 정확히 1개여야 합니다: {cfgs}"
LORA_PATH = os.path.dirname(cfgs[0])
print("어댑터:", LORA_PATH)
assert "v2" in LORA_PATH.lower(), "v2 어댑터가 아닙니다!"

---
## [5] 답 추출기 ▶️

In [ ]:
import re
from collections import Counter

def extract_boxed(text):
    """Return the raw content inside the LAST \\boxed{...}, brace-balanced."""
    idx = text.rfind('\\boxed')
    if idx == -1:
        return None
    i = idx + len('\\boxed')
    while i < len(text) and text[i] == ' ':
        i += 1
    if i >= len(text):
        return None
    if text[i] != '{':                       # bare form: \boxed 15
        m = re.match(r'-?[\d,]+', text[i:])
        return m.group(0) if m else None
    depth, start = 0, i + 1
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None

def to_int(s):
    """LaTeX/text -> python int, or None. Never uses float(), so huge ints survive."""
    if s is None:
        return None
    s = str(s).strip()
    s = s.replace('{,}', '').replace('{\\,}', '')          # LaTeX thousands separator
    s = re.sub(r'\\(?:text|mathrm|mbox|textbf|textrm)\s*\{([^{}]*)\}', r'\1', s)
    for junk in ['\\!', '\\,', '\\;', '\\:', '\\ ', '\\left', '\\right',
                 '\\$', '$', '%', '~', '^\\circ', '\\%']:
        s = s.replace(junk, '')
    s = s.replace(',', '').replace(' ', '').strip()
    s = re.sub(r'[a-zA-Z]+$', '', s)                       # trailing unit: 42cm -> 42
    while len(s) > 1 and s[0] == '(' and s[-1] == ')':     # (\frac{100}{4}) -> \frac{100}{4}
        s = s[1:-1].strip()
    s = s.rstrip('.')
    if not s:
        return None
    m = re.fullmatch(r'\\[dt]?frac\{([-+]?\d+)\}\{([-+]?\d+)\}', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)/([-+]?\d+)', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)(?:\\times|\\cdot)10\^\{?(\d+)\}?', s)
    if m:
        return int(m.group(1)) * 10 ** int(m.group(2))
    if re.fullmatch(r'[-+]?\d+', s):
        return int(s)
    m = re.fullmatch(r'([-+]?\d+)\.0*', s)
    if m:
        return int(m.group(1))
    return None

def last_int(text):
    for c in reversed(re.findall(r'-?\d[\d,]*', text)):
        v = to_int(c)
        if v is not None:
            return v
    return None

def parse_answer(text):
    """None means 'this sample produced no usable integer' -> dropped from voting."""
    raw = extract_boxed(text)
    if raw is not None:
        return to_int(raw)          # boxed present but unparseable -> None, do NOT guess
    m = re.findall(r'(?:answer|Answer|ANSWER)\s*(?:is|:|=)+\s*\$?(-?[\d,]+)', text)
    if m:
        v = to_int(m[-1])
        if v is not None:
            return v
    return last_int(text)

def majority_vote(values, fallback=0):
    vals = [v for v in values if v is not None]
    if not vals:
        return fallback
    return Counter(vals).most_common(1)[0][0]

_c=[(r"\boxed{132}",132),(r"\boxed{-2,025,078}",-2025078),(r"\boxed{\frac{7}{2}}",None)]
print("parser FAILURES:", sum(parse_answer(t)!=w for t,w in _c), "/", len(_c))

---
## [6] 프롬프트 ▶️ 기준선과 동일

In [ ]:
from transformers import AutoTokenizer

SYSTEM = ("You are an expert competition mathematician. Solve the problem step by step, "
          "concisely. The final answer is ALWAYS a single integer. "
          "End your response with the final integer inside \\boxed{}.")

tok = AutoTokenizer.from_pretrained(MODEL_ID)
prompts = [tok.apply_chat_template(
    [{"role":"system","content":SYSTEM},{"role":"user","content":q}],
    tokenize=False, add_generation_prompt=True) for q in work["question"]]
print(len(prompts), "프롬프트 준비")

---
## [7] 모델 로드 ▶️

`enable_lora=True`로 올리면 **어댑터를 붙였다 뗐다** 할 수 있습니다.
`generate()`에 `lora_request`를 주면 v2, 안 주면 베이스입니다. 한 세션에서 둘 다 가능합니다.

In [ ]:
import time, numpy as np
from collections import Counter
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(model=MODEL_ID, dtype="half", max_model_len=4096,
          gpu_memory_utilization=0.90, tensor_parallel_size=1,
          seed=SEED, trust_remote_code=True,
          enable_lora=True, max_lora_rank=LORA_RANK)
sp = SamplingParams(n=N_SAMPLES, temperature=TEMP, top_p=0.95,
                    max_tokens=MAX_TOKENS, seed=SEED)
print("로드 완료")

---
## [8] 베이스 생성 ▶️ 약 52분

In [ ]:
t0 = time.time()
outs_base = llm.generate(prompts, sp)              # lora_request 없음 = 베이스
print(f"생성 {(time.time()-t0)/60:.1f}분")
vals_base = [[parse_answer(c.text) for c in o.outputs] for o in outs_base]
print("파싱 완료")

---
## [9] v2 생성 ▶️ 약 52분

In [ ]:
t0 = time.time()
outs_v2 = llm.generate(prompts, sp, lora_request=LoRARequest("v2", 1, LORA_PATH))
print(f"생성 {(time.time()-t0)/60:.1f}분")
vals_v2 = [[parse_answer(c.text) for c in o.outputs] for o in outs_v2]
print("파싱 완료")

---
## [10] 조합별 비교 ▶️ (GPU 미사용, 몇 초)

이미 생성한 답들을 여러 방식으로 묶어 봅니다. **추가 생성이 없으므로 공짜입니다.**

### 볼 것
1. `base32`가 0.7400 근처인가 → 환경 재현 확인
2. **`pass@64`(합집합)가 0.8733보다 높은가** → 두 모델이 서로 다른 문제를 푼다는 증거
3. **어떤 조합의 `maj`가 0.7400을 넘는가** → 채택 후보

In [ ]:
def maj(vals, fb=0):
    v = [x for x in vals if x is not None]
    return Counter(v).most_common(1)[0][0] if v else fb

def ev(lists, label):
    m  = [int(maj(v)) == int(g) for v, g in zip(lists, gold)]
    p  = [any(x is not None and int(x) == int(g) for x in v) for v, g in zip(lists, gold)]
    sh = [Counter([x for x in v if x is not None]).most_common(1)[0][1]/len(v)
          if any(x is not None for x in v) else 0 for v in lists]
    fl = np.mean([x is None for v in lists for x in v])
    n  = len(lists[0])
    print(f"{label:<16} n={n:>2}  maj={np.mean(m):.4f}  pass={np.mean(p):.4f}  "
          f"득표율={np.mean(sh):.3f}  파싱실패={fl:.2%}")
    return {"label": label, "maj": np.mean(m), "pass": np.mean(p), "m": m}

combos = {
    "base only":      [b            for b    in vals_base],
    "v2 only":        [v            for v    in vals_v2],
    "base32+v2_32":   [b + v        for b, v in zip(vals_base, vals_v2)],
    "base32+v2_16":   [b + v[:16]   for b, v in zip(vals_base, vals_v2)],
    "base16+v2_32":   [b[:16] + v   for b, v in zip(vals_base, vals_v2)],
    "base32+v2_8":    [b + v[:8]    for b, v in zip(vals_base, vals_v2)],
}

print("=" * 78)
print("기준선(8/14 측정)   n=32  maj=0.7400  pass=0.8633  득표율=0.726  파싱실패=1.70%")
print("=" * 78)
res = {k: ev(v, k) for k, v in combos.items()}

---
## [11] 결론 + 유의성 ▶️

같은 300문제를 두 방식이 각각 풀었으므로 **대응 비교**가 됩니다.
"조합만 맞힌 문제"와 "베이스만 맞힌 문제"를 세면 단순 정확도 차이보다 확실한 판단이 됩니다.

**McNemar z > 1.96**이면 통계적으로 유의미한 차이입니다.

In [ ]:
import math

base = res["base only"]
best = max((r for k, r in res.items() if k != "base only"), key=lambda r: r["maj"])

print(f"베이스        maj = {base['maj']:.4f}")
print(f"최고 조합     maj = {best['maj']:.4f}  ({best['label']})")
print(f"차이          {(best['maj']-base['maj'])*100:+.2f}%p")

win  = sum((not a) and b for a, b in zip(base["m"], best["m"]))
lose = sum(a and (not b) for a, b in zip(base["m"], best["m"]))
print(f"\n조합만 맞힘: {win}개 / 베이스만 맞힘: {lose}개 / 순증 {win-lose:+d}")
if win + lose >= 10:
    z = abs(win - lose) / math.sqrt(win + lose)
    print(f"McNemar z = {z:.2f}  ({'유의미' if z > 1.96 else '노이즈 범위'})")

print(f"\n리더보드 환산(x0.63) → 예상 {0.78580 + (best['maj']-0.7400)*0.63:.5f}")
print("\n※ 채택 시 8/31 추론 시간은 2배(문제당 64샘플)가 됩니다. 시간 예산 재계산 필요")